# Notebook 01: Document Exploration
Explore parsed documents, analyze chunk quality, and tune chunking parameters.

In [ ]:
import sys
sys.path.insert(0, '..')

from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

from src.parsing.document_parser import DocumentParserFactory, ParseStrategy
from src.parsing.chunking import ChunkerFactory, ChunkingStrategy

print('Imports OK')

In [ ]:
# Parse a sample document
doc_path = Path('../data/sample_docs/faq.md')
doc = DocumentParserFactory.parse(doc_path, strategy=ParseStrategy.RULE_BASED)

print(f'Title: {doc.title}')
print(f'Content length: {len(doc.content)} chars')
print(f'Sections: {len(doc.sections)}')
print(f'Content preview:\n{doc.content[:500]}')

In [ ]:
# Compare chunking strategies
strategies = [
    ChunkingStrategy.FIXED_SIZE,
    ChunkingStrategy.SENTENCE,
    ChunkingStrategy.HIERARCHICAL,
    ChunkingStrategy.SLIDING_WINDOW,
]

results = {}
for strategy in strategies:
    chunker = ChunkerFactory.get_chunker(strategy, chunk_size=512)
    chunks = chunker.chunk(doc)
    results[strategy.value] = {
        'num_chunks': len(chunks),
        'avg_length': sum(len(c.content) for c in chunks) / len(chunks) if chunks else 0,
        'min_length': min(len(c.content) for c in chunks) if chunks else 0,
        'max_length': max(len(c.content) for c in chunks) if chunks else 0,
    }

df = pd.DataFrame(results).T
print(df.round(1))

In [ ]:
# Visualize chunk length distributions
fig, axes = plt.subplots(2, 2, figsize=(12, 8))

for ax, strategy in zip(axes.flatten(), strategies):
    chunker = ChunkerFactory.get_chunker(strategy, chunk_size=512)
    chunks = chunker.chunk(doc)
    lengths = [len(c.content) for c in chunks]
    
    ax.hist(lengths, bins=20, edgecolor='black', alpha=0.7)
    ax.set_title(f'{strategy.value} (n={len(chunks)})')
    ax.set_xlabel('Chunk Length (chars)')
    ax.set_ylabel('Count')

plt.tight_layout()
plt.savefig('../data/processed/chunk_distributions.png', dpi=150)
plt.show()

In [ ]:
# Inspect hierarchical chunks
chunker = ChunkerFactory.get_chunker(ChunkingStrategy.HIERARCHICAL, chunk_size=512)
chunks = chunker.chunk(doc)

for i, chunk in enumerate(chunks[:5]):
    print(f'--- Chunk {i+1} ---')
    print(f'Section: {chunk.section_title}')
    print(f'Tokens: {chunk.token_count}')
    print(f'Content: {chunk.content[:200]}...')
    print()